In [20]:
from pyspark.sql.functions import col
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType


## Q1 — Roles of Driver, Cluster Manager, Executor

Driver: Runs your main program, builds the DAG of transformations, splits it into stages/tasks, schedules those tasks, and collects final results.

Cluster Manager: Negotiates and allocates resources (CPU/memory) across the cluster (Standalone, YARN, Kubernetes, Mesos).

Executor: JVM processes on worker nodes that actually run the tasks and cache data partitions in memory/disk, reporting status back to the Driver.

In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Week6_TransactionPipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print(f"Default parallelism (executors/cores): {spark.sparkContext.defaultParallelism}")

Spark version: 4.1.2
Default parallelism (executors/cores): 8


## Q2 — How Lazy Evaluation Improves Performance
Transformations (select, filter, withColumn, groupBy) are not executed when called — Spark only records them as a logical plan (the DAG). Execution starts only when an action (show(), count(), collect(), write()) is called. Because the optimizer sees the entire chain before running anything, it can combine steps, push filters down to the data source, reorder operations, and prune unused columns — instead of materializing every intermediate DataFrame. This means far fewer passes over the data and less unnecessary shuffling on large datasets.


## Q3 — Read CSV with Header + inferSchema

In [22]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Data/synthetic_transactions_5000.csv")
)

In [23]:
df_infer = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Data/synthetic_transactions_5000.csv")
)
df_infer.printSchema()

root
 |-- userid: string (nullable = true)
 |-- transaction: string (nullable = true)
 |-- region: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sell_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_times: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- storeid: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- base_price: integer (nullable = true)




## Q4 — CSV vs Parquet: Storage & Performance
CSV is row-based: reading even a single column means reading every byte of every row, with no built-in compression, type info, or statistics — and files are larger on disk.
Parquet is columnar: each column's data is stored contiguously, compressed, and annotated with per-row-group min/max statistics. Spark can read only the columns actually referenced (column pruning) and skip whole row-groups that can't match a filter (predicate pushdown) — making both storage and I/O far more efficient for analytical workloads.

## Q5 — Select product_id, price Where Category = 'Electronics'

In [24]:
electronics_df = (
    df.select("product_id", "price")
      .filter(col("product_category") == "Electronics")
)
electronics_df.show(5)

+----------+------+
|product_id| price|
+----------+------+
|     P4887|3876.8|
|     P9070|4928.0|
|     P8309|1360.0|
|     P8005| 921.0|
|     P1041|1554.3|
+----------+------+
only showing top 5 rows


## Q6 — Rename Column & Cast price to Double

In [25]:
df_revised = (
    df.withColumnRenamed("old_name", "new_name")
      .withColumn("price", col("price").cast("double"))
)

In [26]:
df_revised = (
    df
    .withColumnRenamed("sell_amount", "total_amount")
    .withColumn("price", col("price").cast("double"))
)

## Q7 — Lineage Graph (DAG) & Fault Tolerance
The DAG records exactly how each partition of a DataFrame was derived — the source data plus the sequence of transformations applied. If a worker node fails mid-job, Spark doesn't need a checkpoint of that data; it simply recomputes the lost partitions by replaying the recorded lineage on another executor, using the original source data. This is why Spark doesn't need to constantly replicate intermediate data the way some other systems do.

## Q8 — Filter: Status = 'Completed' AND Amount > 1000

In [27]:
completed_high_value = df.filter((col("status") == "Completed") & (col("sell_amount") > 1000))
completed_high_value.show(5)
print(f"Matching rows: {completed_high_value.count()}")

+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+------------------+--------+------+-------+--------+--------+----------+--------+----------+
|userid|transaction|region|transaction_date|product_category|sell_amount|   status|     city|age|subscription|          raw_times|             email|username| price|storeid|quantity|discount|product_id|priority|base_price|
+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+------------------+--------+------+-------+--------+--------+----------+--------+----------+
|U00003|  TXN100003|  East|      2025-05-23|         Fashion|     5490.0|Completed|Bengaluru| 38|       Basic|2025-05-23 22:51:00| user3@example.com|   user3|1098.0|   S059|       5|      25|     P3255|  Urgent|      1464|
|U00004|  TXN100004|  East|      2025-11-02|            Home|     2976.0|Completed|     Pune| 66|     Premiu

## Q9 — Predicate Pushdown in Parquet
When a .filter() is applied to a Parquet read, Spark pushes that filter condition down into the file-scan layer itself. Since each Parquet row-group stores min/max statistics per column, Spark can skip whole row-groups that provably can't satisfy the filter — without decompressing or deserializing them at all. This means much less data is loaded into memory compared to reading everything first and filtering afterward in-memory (the default behavior with CSV).

In [ ]:
pushdown_df = spark.read.parquet("output_parquet").filter(col("region") == "West")
pushdown_count = pushdown_df.count()

## Q10 — Add final_price = base_price * 1.18

In [30]:
from pyspark.sql.functions import round as spark_round

df_revised = df_revised.withColumn(
    "final_price", spark_round(col("base_price") * 1.18, 2)
)
df_revised.select("product_id", "base_price", "final_price").show(5)

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|     P8953|      3065|     3616.7|
|     P2984|      4072|    4804.96|
|     P3255|      1464|    1727.52|
|     P2188|       930|     1097.4|
|     P8060|       351|     414.18|
+----------+----------+-----------+
only showing top 5 rows


## Q11 — Transformations vs Actions
Transformations (lazy, build the DAG, return a new DataFrame): .filter(), .select(), .withColumn(), .groupBy(), .join().
Actions (trigger actual execution, return a value or write data): .show(), .count(), .collect(), .write.save().

## Q12 — Read Parquet, Drop Null user_id, Save as CSV

In [ ]:
import os
print(os.listdir("output_csv_from_parquet"))

result_df = spark.read.option("header", True).csv("output_csv_from_parquet")
result_df.show(5)
print(result_df.count())

In [ ]:
df_clean = df_revised.filter(col("userid").isNotNull() & col("total_amount").isNotNull())
print(f"Row count after dropping null userid/total_amount: {df_clean.count()}")

## Q13 — Client Mode vs Cluster Mode
Client mode: The Driver runs on the machine that submitted the job (e.g. your laptop or an edge node). Good for interactive work like notebooks, but if that machine disconnects, the job dies.
Cluster mode: The Driver itself is launched inside the cluster (on a worker node) by the Cluster Manager. Better for production/unattended jobs, since the submitting machine can disconnect after launch.

## Q14 — Filter: Region = 'North' OR Priority = 'High'

In [33]:
region_or_priority = df.filter((col("region") == "North") | (col("priority") == "High"))
region_or_priority.show(5)
print(f"Matching rows: {region_or_priority.count()}")

+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+------------------+--------+------+-------+--------+--------+----------+--------+----------+
|userid|transaction|region|transaction_date|product_category|sell_amount|   status|     city|age|subscription|          raw_times|             email|username| price|storeid|quantity|discount|product_id|priority|base_price|
+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+------------------+--------+------+-------+--------+--------+----------+--------+----------+
|U00005|  TXN100005| North|      2025-10-24|            Home|     1193.4| Refunded|    Delhi| 52|       Basic|2025-10-24 16:17:00| user5@example.com|   user5|298.35|   S087|       4|      15|     P8060|  Medium|       351|
|U00006|  TXN100006| South|      2025-03-31|           Books|     1179.8|  Pending|Hyderabad| 64|        Fre

In [34]:
region_or_priority = df.filter((col("region") == "West") | (col("priority") == "Urgent"))
region_or_priority.show(5)
print(f"Matching rows: {region_or_priority.count()}")

+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+------+-------+--------+--------+----------+--------+----------+
|userid|transaction|region|transaction_date|product_category|sell_amount|   status|     city|age|subscription|          raw_times|            email|username| price|storeid|quantity|discount|product_id|priority|base_price|
+------+-----------+------+----------------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+------+-------+--------+--------+----------+--------+----------+
|U00001|  TXN100001|  West|      2025-06-09|         Grocery|     5517.0| Refunded|   Jaipur| 58|       Basic|2025-06-09 01:52:00|user1@example.com|   user1|2758.5|   S068|       2|      10|     P8953|     Low|      3065|
|U00002|  TXN100002|  West|      2025-01-04|         Grocery|    27689.6|   Failed|     Pune| 65|       Basic|20

## Q15 — Why .show(5) is Safer than .collect() on Multi-TB Data
.collect() pulls the entire result set from every executor back into the single Driver's memory. On a multi-terabyte dataset, this either crashes the Driver with an out-of-memory error or floods the network shipping huge volumes of data to one process. .show(5) (or .take(n)) only computes and materializes the small number of rows requested, keeping memory usage bounded no matter how large the underlying dataset is.

In [ ]:
# Safe: only pulls 5 rows to the driver
df_clean.show(5)

# Unsafe at scale — never do this on a huge DataFrame:
# all_rows = df_clean.collect()

In [39]:
# ============================================================
# STEP 0 — Imports
# ============================================================
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

# ============================================================
# STEP 1 — Spark session
# ============================================================
spark = (
    SparkSession.builder
    .appName("Week6_TransactionPipeline")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# ============================================================
# STEP 2 — Input file path (CHANGE THIS to your actual file location)
# ============================================================
input_path = r"C:\Users\sadiy\OneDrive\Desktop\WEEK-6_Assignment\Data\synthetic_transactions_5000.csv"

# ============================================================
# STEP 3 — Schema + Read
# ============================================================
schema = StructType([
    StructField("userid", StringType(), True),
    StructField("transaction", StringType(), True),
    StructField("region", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("sell_amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("city", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("subscription", StringType(), True),
    StructField("raw_times", StringType(), True),
    StructField("email", StringType(), True),
    StructField("username", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("storeid", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("base_price", DoubleType(), True),
])

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(input_path)
)

# ============================================================
# STEP 4 — Transform
# ============================================================
df_revised = (
    df
    .withColumnRenamed("sell_amount", "total_amount")
    .withColumn("price", col("price").cast("double"))
    .withColumn("final_price", spark_round(col("base_price") * 1.18, 2))
)

# ============================================================
# STEP 5 — Filter nulls
# ============================================================
df_clean = df_revised.filter(
    col("userid").isNotNull() & col("total_amount").isNotNull()
)

# ============================================================
# STEP 6 — Save as CSV in an "output" folder (no Hadoop needed)
# ============================================================
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "cleaned_transactions.csv")

df_clean.toPandas().to_csv(output_file, index=False)

print("CSV saved to:", os.path.abspath(output_file))

CSV saved to: C:\Users\sadiy\OneDrive\Desktop\WEEK-6_Assignment\output\cleaned_transactions.csv
